# 🛡️ CyberSentinel — Model Experiment Notebook
> Run each model & stage individually. All tunable parameters live in `config/model_params.json`.



In [24]:
import json, os, sys, warnings, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("."))
print("✓ imports loaded")



✓ imports loaded


## ⚙️ Configuration — edit `config/model_params.json` to tune parameters



In [25]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SELECT MODEL — uncomment exactly ONE line below           ║
# ╚══════════════════════════════════════════════════════════════╝

SELECTED_MODEL = "xgboost"
# SELECTED_MODEL = "randomforest"
# SELECTED_MODEL = "svm"

# ╔══════════════════════════════════════════════════════════════╗
# ║  SELECT DATASET                                            ║
# ╚══════════════════════════════════════════════════════════════╝

TRAIN_CSV = os.path.join("data", "train", "KDDTrain+.csv")
# TRAIN_CSV = os.path.join("data", "train", "KDDTrain+_20Percent.csv")  # faster

TEST_CSV  = os.path.join("data", "test", "KDDTest-21.csv")
# TEST_CSV  = os.path.join("data", "test", "KDDTest+.csv")

# ╔══════════════════════════════════════════════════════════════╗
# ║  SELECT MODE — train, test, or both                        ║
# ╚══════════════════════════════════════════════════════════════╝

RUN_TRAIN = True
RUN_TEST  = True

print(f"Model : {SELECTED_MODEL}")
print(f"Train : {TRAIN_CSV}")
print(f"Test  : {TEST_CSV}")
# Output directory for saved models
out_dir = os.path.join("model","cascade",SELECTED_MODEL.lower())
os.makedirs(out_dir, exist_ok=True)



Model : xgboost
Train : data\train\KDDTrain+.csv
Test  : data\test\KDDTest-21.csv


In [26]:
# ── Load tunable parameters from JSON ──
with open(os.path.join("config", "model_params.json")) as f:
    PARAMS = json.load(f)

GENERAL      = PARAMS["general"]                         # val_size, random_state, etc.
MODEL_PARAMS = PARAMS[SELECTED_MODEL]                    # per-model stage1/stage2 hyperparams
CASCADE_TH   = PARAMS["cascade_thresholds"]              # threshold grid ranges
DROP_COLS    = PARAMS["drop_columns"]                     # columns to remove

print("\n── General ──")
for k,v in GENERAL.items():
    if not k.startswith("_"): print(f"  {k}: {v}")

print(f"\n── {SELECTED_MODEL} Stage 1 hyperparams ──")
for k,v in MODEL_PARAMS["stage1"].items():
    if not k.startswith("_"): print(f"  {k}: {v}")

print(f"\n── {SELECTED_MODEL} Stage 2 hyperparams ──")
for k,v in MODEL_PARAMS["stage2"].items():
    if not k.startswith("_"): print(f"  {k}: {v}")

print("\n── Cascade threshold grid ──")
for k,v in CASCADE_TH.items():
    if not k.startswith("_"): print(f"  {k}: {v}")




── General ──
  val_size: 0.2
  random_state: 42
  calibration_method: sigmoid
  min_attack_recall_for_threshold_search: 0.9

── xgboost Stage 1 hyperparams ──
  n_estimators: 400
  max_depth: 5
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_weight: 5
  gamma: 0.3
  reg_alpha: 1.0
  reg_lambda: 2.0
  scale_pos_weight: auto

── xgboost Stage 2 hyperparams ──
  n_estimators: 400
  max_depth: 5
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_weight: 3
  gamma: 0.2
  reg_alpha: 0.5
  reg_lambda: 1.5

── Cascade threshold grid ──
  stage1_low: {'_comment': 'Below this p(attack), sample is classified as Normal immediately', 'start': 0.2, 'stop': 0.76, 'step': 0.05}
  stage1_strong: {'_comment': 'Above this p(attack), sample goes straight to Stage 2 attack classification', 'start': 0.75, 'stop': 0.96, 'step': 0.05}
  stage2_conf: {'_comment': 'Minimum top-1 confidence from Stage 2 to accept an attack prediction', 'start': 0.3, 'stop': 0.

## 🔧 Helper Functions



In [27]:
CATEGORICAL_COLS = ["protocol_type", "service", "flag"]

ATTACK_MAP = {
    "back":"DoS","land":"DoS","neptune":"DoS","pod":"DoS","smurf":"DoS",
    "teardrop":"DoS","apache2":"DoS","mailbomb":"DoS","processtable":"DoS",
    "udpstorm":"DoS","worm":"DoS",
    "satan":"Probe","ipsweep":"Probe","nmap":"Probe","portsweep":"Probe",
    "mscan":"Probe","saint":"Probe",
    "guess_passwd":"R2L","ftp_write":"R2L","imap":"R2L","phf":"R2L",
    "multihop":"R2L","warezmaster":"R2L","warezclient":"R2L","spy":"R2L",
    "xlock":"R2L","xsnoop":"R2L","snmpguess":"R2L","snmpgetattack":"R2L",
    "httptunnel":"R2L","sendmail":"R2L","named":"R2L",
    "buffer_overflow":"U2R","loadmodule":"U2R","rootkit":"U2R","perl":"U2R",
    "sqlattack":"U2R","xterm":"U2R","ps":"U2R",
}

def load_and_clean(path):
    df = pd.read_csv(path).drop_duplicates()
    for c in df.select_dtypes("number").columns:
        df[c].fillna(df[c].median(), inplace=True)
    for c in df.select_dtypes("object").columns:
        df[c].fillna(df[c].mode()[0], inplace=True)
    drop = [c for c in DROP_COLS if c in df.columns]
    if drop: df = df.drop(columns=drop)
    return df

def to_binary(labels):
    return np.array([0 if str(v).strip().lower()=="normal" else 1 for v in labels])

def to_category(labels):
    return pd.Series([ATTACK_MAP.get(str(v).strip().lower(), "R2L") for v in labels],
                     index=labels.index)

def make_preprocessor(X):
    cats = [c for c in CATEGORICAL_COLS if c in X.columns]
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cats)],
                             remainder="passthrough")

def scale_pos_weight(y):
    neg = int((y==0).sum()); pos = int((y==1).sum())
    return max(1.0, neg/pos) if pos else 1.0

def build_pipeline(stage, X, y_bin=None):
    pre = make_preprocessor(X)
    p = MODEL_PARAMS[stage]
    m = SELECTED_MODEL.lower()
    if m == "randomforest":
        clf = RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
              min_samples_split=p["min_samples_split"], min_samples_leaf=p["min_samples_leaf"],
              class_weight=p["class_weight"], random_state=GENERAL["random_state"], n_jobs=-1)
        return Pipeline([("preprocessor",pre),("classifier",clf)])
    if m == "svm":
        clf = LinearSVC(C=p["C"], class_weight=p["class_weight"], dual=p["dual"],
              max_iter=p["max_iter"], random_state=GENERAL["random_state"])
        return Pipeline([("preprocessor",pre),("scaler",StandardScaler(with_mean=False)),("classifier",clf)])
    if m == "xgboost":
        kw = dict(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
              learning_rate=p["learning_rate"], subsample=p["subsample"],
              colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
              gamma=p["gamma"], reg_alpha=p["reg_alpha"], reg_lambda=p["reg_lambda"],
              random_state=GENERAL["random_state"], n_jobs=-1)
        if stage == "stage1":
            spw = p.get("scale_pos_weight","auto")
            kw["scale_pos_weight"] = scale_pos_weight(y_bin) if spw=="auto" else spw
            kw["eval_metric"] = "logloss"
        else:
            kw["eval_metric"] = "mlogloss"
            kw["objective"] = "multi:softprob"
        clf = XGBClassifier(**kw)
        return Pipeline([("preprocessor",pre),("classifier",clf)])
    raise ValueError(f"Unknown model: {m}")

print("✓ helpers defined")



✓ helpers defined


## 📂 Load & Prepare Data

Using an internal 80/20 split from the training CSV. The 80% split is used for train/val/cal, and the 20% split is the holdout test set.


In [28]:
df_full = load_and_clean(TRAIN_CSV)
print(f"Training data: {len(df_full)} rows, {df_full.shape[1]} cols")

y_binary_all = to_binary(df_full["label"])

# 80/20 split: internal train vs holdout test
train_main_df, holdout_test_df = train_test_split(
    df_full,
    test_size=0.2,
    random_state=GENERAL["random_state"],
    stratify=y_binary_all,
)

y_train_main_bin = to_binary(train_main_df["label"])

# 3-way split from the 80% training portion: train / calibration / validation
# Calibration set is used ONLY for probability calibration
# Validation set is used ONLY for metrics & threshold tuning (no leakage)
train_df, temp_df = train_test_split(
    train_main_df,
    test_size=GENERAL["val_size"],
    random_state=GENERAL["random_state"],
    stratify=y_train_main_bin,
)
y_temp_bin = to_binary(temp_df["label"])
cal_df, val_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=GENERAL["random_state"],
    stratify=y_temp_bin,
)

train_df = train_df.reset_index(drop=True)
cal_df = cal_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
holdout_test_df = holdout_test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Cal: {len(cal_df)}  |  Val: {len(val_df)}")
print(f"Holdout test: {len(holdout_test_df)}")


Training data: 125973 rows, 31 cols
Train: 80622  |  Cal: 10078  |  Val: 10078
Holdout test: 25195


## 🔷 Stage 1 — Binary Classification (Normal vs Attack)



In [29]:
if RUN_TRAIN:
    X_tr1 = train_df.drop(columns=["label","attack_category"], errors="ignore")
    y_tr1 = to_binary(train_df["label"])
    X_v1  = val_df.drop(columns=["label","attack_category"], errors="ignore")
    y_v1  = to_binary(val_df["label"])

    pipe1 = build_pipeline("stage1", X_tr1, y_tr1)
    pipe1.fit(X_tr1, y_tr1)

    pred1 = pipe1.predict(X_v1)
    acc1  = accuracy_score(y_v1, pred1)
    cm1   = confusion_matrix(y_v1, pred1, labels=[0,1])
    tn,fp,fn,tp = cm1.ravel()

    print(f"\n{'='*55}")
    print(f"  STAGE 1 RESULTS \u2014 {SELECTED_MODEL.upper()}")
    print(f"{'='*55}")
    print(f"  Accuracy     : {acc1*100:.2f}%")
    print(f"  F1           : {f1_score(y_v1, pred1):.4f}")
    ar1 = tp/(tp+fn) if (tp+fn) else 0
    fpr1 = fp/(fp+tn) if (fp+tn) else 0
    print(f"  Attack Recall: {ar1*100:.2f}%")
    print(f"  FPR (Normal) : {fpr1*100:.4f}%")
    print(f"  Confusion    : TN={tn} FP={fp} FN={fn} TP={tp}")
    print(classification_report(y_v1, pred1, target_names=["Normal","Attack"]))

    # Save stage 1
    joblib.dump(pipe1, os.path.join(out_dir,"stage1_model.pkl"))
    print(f"\u2713 Stage 1 model saved \u2192 {out_dir}/stage1_model.pkl")
else:
    print("\u23ed Skipping Stage 1 training (RUN_TRAIN=False)")




  STAGE 1 RESULTS — XGBOOST
  Accuracy     : 99.89%
  F1           : 0.9988
  Attack Recall: 99.85%
  FPR (Normal) : 0.0742%
  Confusion    : TN=5384 FP=4 FN=7 TP=4683
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      5388
      Attack       1.00      1.00      1.00      4690

    accuracy                           1.00     10078
   macro avg       1.00      1.00      1.00     10078
weighted avg       1.00      1.00      1.00     10078

✓ Stage 1 model saved → model\cascade\xgboost/stage1_model.pkl


## 🔶 Stage 2 — Attack Category Classification (DoS / Probe / R2L / U2R)



In [30]:
if RUN_TRAIN:
    y_tr_bin = to_binary(train_df["label"])
    y_v_bin  = to_binary(val_df["label"])

    atk_tr = train_df[y_tr_bin==1].copy()
    atk_v  = val_df[y_v_bin==1].copy()

    atk_tr["attack_category"] = to_category(atk_tr["label"])
    atk_v["attack_category"]  = to_category(atk_v["label"])

    X_tr2 = atk_tr.drop(columns=["label","attack_category"], errors="ignore")
    X_v2  = atk_v.drop(columns=["label","attack_category"], errors="ignore")

    le2 = LabelEncoder()
    y_tr2 = le2.fit_transform(atk_tr["attack_category"])
    y_v2  = le2.transform(atk_v["attack_category"])

    pipe2 = build_pipeline("stage2", X_tr2)

    # Fit with sample weights for class balancing
    classes, counts = np.unique(y_tr2, return_counts=True)
    cw = {c: len(y_tr2)/(len(classes)*n) for c,n in zip(classes,counts)}
    sw = np.array([cw[v] for v in y_tr2])
    pipe2.fit(X_tr2, y_tr2, classifier__sample_weight=sw)

    pred2 = pipe2.predict(X_v2)
    acc2  = accuracy_score(y_v2, pred2)
    f1_m  = f1_score(y_v2, pred2, average="macro", zero_division=0)

    print(f"\n{'='*55}")
    print(f"  STAGE 2 RESULTS — {SELECTED_MODEL.upper()}")
    print(f"{'='*55}")
    print(f"  Accuracy : {acc2*100:.2f}%")
    print(f"  Macro F1 : {f1_m:.4f}")
    print(f"  Classes  : {list(le2.classes_)}")
    print(classification_report(y_v2, pred2, target_names=le2.classes_, zero_division=0))

    # Save stage 2
    joblib.dump(pipe2, os.path.join(out_dir,"stage2_model.pkl"))
    joblib.dump(le2,   os.path.join(out_dir,"stage2_label_encoder.pkl"))
    print(f"✓ Stage 2 model + encoder saved → {out_dir}")
else:
    print("⏭ Skipping Stage 2 training (RUN_TRAIN=False)")




  STAGE 2 RESULTS — XGBOOST
  Accuracy : 100.00%
  Macro F1 : 1.0000
  Classes  : ['DoS', 'Probe', 'R2L', 'U2R']
              precision    recall  f1-score   support

         DoS       1.00      1.00      1.00      3696
       Probe       1.00      1.00      1.00       904
         R2L       1.00      1.00      1.00        87
         U2R       1.00      1.00      1.00         3

    accuracy                           1.00      4690
   macro avg       1.00      1.00      1.00      4690
weighted avg       1.00      1.00      1.00      4690

✓ Stage 2 model + encoder saved → model\cascade\xgboost


## 🎯 Calibration — Probability Calibration for Cascade



In [31]:
if RUN_TRAIN:
    method = GENERAL["calibration_method"]

    def calibrate(base, X_cal, y_cal, method):
        try:
            from sklearn.frozen import FrozenEstimator
            cal = CalibratedClassifierCV(FrozenEstimator(base), method=method, cv=None)
        except Exception:
            cal = CalibratedClassifierCV(base, method=method, cv="prefit")
        cal.fit(X_cal, y_cal)
        return cal

    # ── Calibrate Stage 1 using dedicated calibration split ──
    X_cal1 = cal_df.drop(columns=["label","attack_category"], errors="ignore")
    y_cal1 = to_binary(cal_df["label"])
    cal1 = calibrate(pipe1, X_cal1, y_cal1, method)
    joblib.dump(cal1, os.path.join(out_dir,"stage1_calibrated.pkl"))

    # ── Calibrate Stage 2 — attack-only samples from cal split ──
    atk_cal = cal_df[to_binary(cal_df["label"]) == 1].copy()
    atk_cal["attack_category"] = to_category(atk_cal["label"])
    X_cal2 = atk_cal.drop(columns=["label","attack_category"], errors="ignore")
    y_cal2 = le2.transform(atk_cal["attack_category"])
    cal2 = calibrate(pipe2, X_cal2, y_cal2, method)
    joblib.dump(cal2, os.path.join(out_dir,"stage2_calibrated.pkl"))

    print(f"\u2713 Both stages calibrated ({method}) using dedicated cal split")
    print(f"  Cal set: {len(cal_df)} total, {len(atk_cal)} attacks")
else:
    print("\u23ed Skipping calibration (RUN_TRAIN=False)")



✓ Both stages calibrated (sigmoid) using dedicated cal split
  Cal set: 10078 total, 4691 attacks


## 🔥 Simulated Annealing — Joint RF + Threshold Search

This runs a constrained simulated annealing search focused on maximizing recall, then minimizing FPR, then precision, then accuracy, with an extra FNR penalty. Results are logged to CSV and the best config is saved to JSON for reproducibility.


### SA Documentation and Space Mapping

- Objective (maximize): $0.45R + 0.25(1 - FPR) + 0.20P + 0.10A - k \cdot FNR$, with $k = 0.30$.
- Schedule: initial temp = 10, cooling = 0.95, iterations = 200, 1 neighbor/iter.
- Search space: bounded, discrete steps for RandomForest stage1+stage2 + cascade thresholds.
- Outputs: a full per-iteration log (CSV), best config (JSON), and the search space definition (JSON).


In [32]:
# Simulated annealing search (RandomForest + cascade thresholds)
# Note: This is a new optimization algorithm, added per request.
if RUN_TRAIN:
    model_name = "randomforest"
    rf_out_dir = os.path.join("model", "cascade", model_name)
    os.makedirs(rf_out_dir, exist_ok=True)

    SA_CONFIG = {
        "initial_temp": 10.0,
        "cooling_rate": 0.95,
        "iterations": 20,
        "neighbors": 1,
        "fnr_penalty": 0.30,
        "log_every": 10,
    }

    SA_WEIGHTS = {
        "recall": 0.45,
        "inv_fpr": 0.25,
        "precision": 0.20,
        "accuracy": 0.10,
    }

    search_space = {
        "rf_stage1": {
            "n_estimators": {"min": 200, "max": 600, "step": 50, "type": "int"},
            "max_depth": {"min": 6, "max": 20, "step": 2, "type": "int"},
            "min_samples_split": {"min": 2, "max": 12, "step": 2, "type": "int"},
            "min_samples_leaf": {"min": 1, "max": 6, "step": 1, "type": "int"},
            "class_weight": {"choices": ["balanced_subsample", "balanced", None]},
        },
        "rf_stage2": {
            "n_estimators": {"min": 200, "max": 600, "step": 50, "type": "int"},
            "max_depth": {"min": 6, "max": 20, "step": 2, "type": "int"},
            "min_samples_split": {"min": 2, "max": 12, "step": 2, "type": "int"},
            "min_samples_leaf": {"min": 1, "max": 6, "step": 1, "type": "int"},
            "class_weight": {"choices": ["balanced_subsample", "balanced", None]},
        },
        "thresholds": {
            "stage1_low": {
                "min": float(CASCADE_TH["stage1_low"]["start"]),
                "max": float(CASCADE_TH["stage1_low"]["stop"]),
                "step": float(CASCADE_TH["stage1_low"]["step"]),
                "type": "float",
            },
            "stage1_strong": {
                "min": float(CASCADE_TH["stage1_strong"]["start"]),
                "max": float(CASCADE_TH["stage1_strong"]["stop"]),
                "step": float(CASCADE_TH["stage1_strong"]["step"]),
                "type": "float",
            },
            "stage2_conf": {
                "min": float(CASCADE_TH["stage2_conf"]["start"]),
                "max": float(CASCADE_TH["stage2_conf"]["stop"]),
                "step": float(CASCADE_TH["stage2_conf"]["step"]),
                "type": "float",
            },
            "stage2_margin": {
                "min": float(CASCADE_TH["stage2_margin"]["start"]),
                "max": float(CASCADE_TH["stage2_margin"]["stop"]),
                "step": float(CASCADE_TH["stage2_margin"]["step"]),
                "type": "float",
            },
        },
    }

    space_map_path = os.path.join(rf_out_dir, "sa_search_space_randomforest.json")
    with open(space_map_path, "w") as f:
        json.dump(
            {
                "search_space": search_space,
                "weights": SA_WEIGHTS,
                "sa_config": SA_CONFIG,
            },
            f,
            indent=2,
        )

    def _snap(value, lo, hi, step, is_int):
        if step <= 0:
            v = max(lo, min(hi, value))
        else:
            v = lo + round((value - lo) / step) * step
            v = max(lo, min(hi, v))
        return int(round(v)) if is_int else float(v)

    def _midpoint(param):
        return _snap((param["min"] + param["max"]) / 2.0, param["min"], param["max"], param["step"], False)

    def _build_rf_pipeline(params, X):
        pre = make_preprocessor(X)
        clf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            class_weight=params["class_weight"],
            random_state=GENERAL["random_state"],
            n_jobs=-1,
        )
        return Pipeline([("preprocessor", pre), ("classifier", clf)])

    def _calibrate(base, X_cal, y_cal, method):
        try:
            from sklearn.frozen import FrozenEstimator
            cal = CalibratedClassifierCV(FrozenEstimator(base), method=method, cv=None)
        except Exception:
            cal = CalibratedClassifierCV(base, method=method, cv="prefit")
        cal.fit(X_cal, y_cal)
        return cal

    def _normalize_thresholds(thresholds):
        if thresholds["stage1_strong"] <= thresholds["stage1_low"]:
            step = search_space["thresholds"]["stage1_strong"]["step"]
            thresholds["stage1_strong"] = min(
                thresholds["stage1_low"] + step,
                search_space["thresholds"]["stage1_strong"]["max"],
            )
        return thresholds

    def _init_config():
        rf_default = PARAMS[model_name]
        cfg = {
            "rf_stage1": dict(rf_default["stage1"]),
            "rf_stage2": dict(rf_default["stage2"]),
            "thresholds": {
                "stage1_low": _midpoint(search_space["thresholds"]["stage1_low"]),
                "stage1_strong": _midpoint(search_space["thresholds"]["stage1_strong"]),
                "stage2_conf": _midpoint(search_space["thresholds"]["stage2_conf"]),
                "stage2_margin": _midpoint(search_space["thresholds"]["stage2_margin"]),
            },
        }
        cfg["thresholds"] = _normalize_thresholds(cfg["thresholds"])
        return cfg

    def _neighbor(cfg, rng):
        new_cfg = {
            "rf_stage1": dict(cfg["rf_stage1"]),
            "rf_stage2": dict(cfg["rf_stage2"]),
            "thresholds": dict(cfg["thresholds"]),
        }
        targets = [
            ("rf_stage1", k) for k in search_space["rf_stage1"].keys()
        ] + [
            ("rf_stage2", k) for k in search_space["rf_stage2"].keys()
        ] + [
            ("thresholds", k) for k in search_space["thresholds"].keys()
        ]
        group, key = targets[rng.integers(0, len(targets))]
        spec = search_space[group][key]
        if "choices" in spec:
            new_cfg[group][key] = rng.choice(spec["choices"]).item() if hasattr(rng.choice(spec["choices"]), "item") else rng.choice(spec["choices"])
        else:
            step = spec["step"]
            direction = rng.choice([-1, 1])
            candidate = new_cfg[group][key] + direction * step
            new_cfg[group][key] = _snap(candidate, spec["min"], spec["max"], spec["step"], spec["type"] == "int")
        if group == "thresholds":
            new_cfg["thresholds"] = _normalize_thresholds(new_cfg["thresholds"])
        return new_cfg

    def _evaluate(cfg):
        X_tr1 = train_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_tr1 = to_binary(train_df["label"])
        X_v1 = val_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_v1 = to_binary(val_df["label"])

        pipe1 = _build_rf_pipeline(cfg["rf_stage1"], X_tr1)
        pipe1.fit(X_tr1, y_tr1)

        y_tr_bin = to_binary(train_df["label"])
        atk_tr = train_df[y_tr_bin == 1].copy()
        atk_tr["attack_category"] = to_category(atk_tr["label"])
        X_tr2 = atk_tr.drop(columns=["label", "attack_category"], errors="ignore")
        le2 = LabelEncoder()
        y_tr2 = le2.fit_transform(atk_tr["attack_category"])

        pipe2 = _build_rf_pipeline(cfg["rf_stage2"], X_tr2)
        classes, counts = np.unique(y_tr2, return_counts=True)
        cw = {c: len(y_tr2) / (len(classes) * n) for c, n in zip(classes, counts)}
        sw = np.array([cw[v] for v in y_tr2])
        pipe2.fit(X_tr2, y_tr2, classifier__sample_weight=sw)

        method = GENERAL["calibration_method"]
        X_cal1 = cal_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal1 = to_binary(cal_df["label"])
        cal1 = _calibrate(pipe1, X_cal1, y_cal1, method)

        atk_cal = cal_df[to_binary(cal_df["label"]) == 1].copy()
        atk_cal["attack_category"] = to_category(atk_cal["label"])
        X_cal2 = atk_cal.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal2 = le2.transform(atk_cal["attack_category"])
        cal2 = _calibrate(pipe2, X_cal2, y_cal2, method)

        p_atk = cal1.predict_proba(X_v1)[:, 1]
        tl = cfg["thresholds"]["stage1_low"]
        ts = cfg["thresholds"]["stage1_strong"]
        tc = cfg["thresholds"]["stage2_conf"]
        tm = cfg["thresholds"]["stage2_margin"]

        s2_mask = p_atk >= tl
        pred_cat = np.full(len(X_v1), "", dtype=object)
        top1 = np.zeros(len(X_v1))
        marg = np.zeros(len(X_v1))

        if s2_mask.any():
            p_cat = cal2.predict_proba(X_v1[s2_mask])
            top1[s2_mask] = np.max(p_cat, axis=1)
            top2_vals = np.partition(p_cat, -2, axis=1)[:, -2]
            marg[s2_mask] = top1[s2_mask] - top2_vals
            pred_cat[s2_mask] = le2.inverse_transform(np.argmax(p_cat, axis=1))

        final = []
        for i in range(len(X_v1)):
            if p_atk[i] < tl:
                final.append("normal")
            elif p_atk[i] >= ts:
                final.append(pred_cat[i])
            elif top1[i] < tc or marg[i] < tm:
                final.append("normal")
            else:
                final.append(pred_cat[i])

        yt = y_v1
        yp = np.array([0 if v == "normal" else 1 for v in final])
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        recall = tp / (tp + fn) if (tp + fn) else 0.0
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        accuracy = (tp + tn) / len(yt) if len(yt) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        fnr = fn / (fn + tp) if (fn + tp) else 0.0

        score = (
            SA_WEIGHTS["recall"] * recall
            + SA_WEIGHTS["inv_fpr"] * (1.0 - fpr)
            + SA_WEIGHTS["precision"] * precision
            + SA_WEIGHTS["accuracy"] * accuracy
            - SA_CONFIG["fnr_penalty"] * fnr
        )

        metrics = {
            "score": float(score),
            "recall": float(recall),
            "precision": float(precision),
            "accuracy": float(accuracy),
            "fpr": float(fpr),
            "fnr": float(fnr),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        }
        return metrics

    rng = np.random.default_rng(GENERAL["random_state"])
    current = _init_config()
    current_metrics = _evaluate(current)
    best = dict(current)
    best_metrics = dict(current_metrics)

    rows = []
    rows.append({"iter": 0, "accepted": True, **current_metrics, **current["thresholds"], **current["rf_stage1"], **current["rf_stage2"]})

    temp = SA_CONFIG["initial_temp"]
    for i in range(1, SA_CONFIG["iterations"] + 1):
        for _ in range(SA_CONFIG["neighbors"]):
            candidate = _neighbor(current, rng)
            cand_metrics = _evaluate(candidate)
            delta = cand_metrics["score"] - current_metrics["score"]
            accept = delta > 0 or (temp > 0 and rng.random() < np.exp(delta / temp))
            if accept:
                current = candidate
                current_metrics = cand_metrics

            if cand_metrics["score"] > best_metrics["score"]:
                best = candidate
                best_metrics = cand_metrics

            rows.append({
                "iter": i,
                "accepted": bool(accept),
                **cand_metrics,
                **candidate["thresholds"],
                **candidate["rf_stage1"],
                **candidate["rf_stage2"],
            })

        temp *= SA_CONFIG["cooling_rate"]
        if i % SA_CONFIG["log_every"] == 0:
            print(f"[SA] iter={i} score={best_metrics['score']:.4f} recall={best_metrics['recall']:.4f} fpr={best_metrics['fpr']:.4f}")

    log_path = os.path.join(rf_out_dir, "sa_search_log_randomforest.csv")
    pd.DataFrame(rows).to_csv(log_path, index=False)

    best_out = {
        "model": model_name,
        "objective": {
            "weights": SA_WEIGHTS,
            "fnr_penalty": SA_CONFIG["fnr_penalty"],
        },
        "sa_config": SA_CONFIG,
        "search_space_path": os.path.basename(space_map_path),
        "best_config": best,
        "best_metrics": best_metrics,
    }
    best_path = os.path.join(rf_out_dir, "sa_best_config.json")
    with open(best_path, "w") as f:
        json.dump(best_out, f, indent=2)

    print("\n[SA] Best metrics:")
    for k in ["score", "recall", "fpr", "precision", "accuracy", "fnr"]:
        print(f"  {k:10s}: {best_metrics[k]:.6f}")
    print(f"[SA] Space map saved -> {space_map_path}")
    print(f"[SA] Log saved -> {log_path}")
    print(f"[SA] Best config saved -> {best_path}")
else:
    print("Skipping simulated annealing (RUN_TRAIN=False)")


[SA] iter=10 score=0.9973 recall=0.9974 fpr=0.0013
[SA] iter=20 score=0.9973 recall=0.9974 fpr=0.0013

[SA] Best metrics:
  score     : 0.997269
  recall    : 0.997441
  fpr       : 0.001299
  precision : 0.998506
  accuracy  : 0.998115
  fnr       : 0.002559
[SA] Space map saved -> model\cascade\randomforest\sa_search_space_randomforest.json
[SA] Log saved -> model\cascade\randomforest\sa_search_log_randomforest.csv
[SA] Best config saved -> model\cascade\randomforest\sa_best_config.json


## 🧪 SA Best — Test Evaluation

Rebuilds the RandomForest cascade using the SA best config, calibrates on the cal split, and evaluates on the test set.


In [33]:
if RUN_TEST:
    model_name = "randomforest"
    rf_out_dir = os.path.join("model", "cascade", model_name)
    sa_best_path = os.path.join(rf_out_dir, "sa_best_config.json")

    if not os.path.exists(sa_best_path):
        print("SA best config not found. Run the SA cell first.")
    else:
        with open(sa_best_path) as f:
            sa_cfg = json.load(f)
        cfg = sa_cfg["best_config"]

        def _build_rf_pipeline(params, X):
            pre = make_preprocessor(X)
            clf = RandomForestClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                class_weight=params["class_weight"],
                random_state=GENERAL["random_state"],
                n_jobs=-1,
            )
            return Pipeline([("preprocessor", pre), ("classifier", clf)])

        def _calibrate(base, X_cal, y_cal, method):
            try:
                from sklearn.frozen import FrozenEstimator
                cal = CalibratedClassifierCV(FrozenEstimator(base), method=method, cv=None)
            except Exception:
                cal = CalibratedClassifierCV(base, method=method, cv="prefit")
            cal.fit(X_cal, y_cal)
            return cal

        X_tr1 = train_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_tr1 = to_binary(train_df["label"])
        pipe1 = _build_rf_pipeline(cfg["rf_stage1"], X_tr1)
        pipe1.fit(X_tr1, y_tr1)

        y_tr_bin = to_binary(train_df["label"])
        atk_tr = train_df[y_tr_bin == 1].copy()
        atk_tr["attack_category"] = to_category(atk_tr["label"])
        X_tr2 = atk_tr.drop(columns=["label", "attack_category"], errors="ignore")
        le2 = LabelEncoder()
        y_tr2 = le2.fit_transform(atk_tr["attack_category"])

        pipe2 = _build_rf_pipeline(cfg["rf_stage2"], X_tr2)
        classes, counts = np.unique(y_tr2, return_counts=True)
        cw = {c: len(y_tr2) / (len(classes) * n) for c, n in zip(classes, counts)}
        sw = np.array([cw[v] for v in y_tr2])
        pipe2.fit(X_tr2, y_tr2, classifier__sample_weight=sw)

        method = GENERAL["calibration_method"]
        X_cal1 = cal_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal1 = to_binary(cal_df["label"])
        cal1 = _calibrate(pipe1, X_cal1, y_cal1, method)

        atk_cal = cal_df[to_binary(cal_df["label"]) == 1].copy()
        atk_cal["attack_category"] = to_category(atk_cal["label"])
        X_cal2 = atk_cal.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal2 = le2.transform(atk_cal["attack_category"])
        cal2 = _calibrate(pipe2, X_cal2, y_cal2, method)

        X_test = holdout_test_df.drop(columns=["label", "attack_category"], errors="ignore")
        p_atk = cal1.predict_proba(X_test)[:, 1]
        tl = cfg["thresholds"]["stage1_low"]
        ts = cfg["thresholds"]["stage1_strong"]
        tc = cfg["thresholds"]["stage2_conf"]
        tm = cfg["thresholds"]["stage2_margin"]

        s2_mask = p_atk >= tl
        pred_cat = np.full(len(X_test), "", dtype=object)
        top1 = np.zeros(len(X_test))
        marg = np.zeros(len(X_test))

        if s2_mask.any():
            p_cat = cal2.predict_proba(X_test[s2_mask])
            top1[s2_mask] = np.max(p_cat, axis=1)
            top2_vals = np.partition(p_cat, -2, axis=1)[:, -2]
            marg[s2_mask] = top1[s2_mask] - top2_vals
            pred_cat[s2_mask] = le2.inverse_transform(np.argmax(p_cat, axis=1))

        final, path = [], []
        for i in range(len(X_test)):
            if p_atk[i] < tl:
                final.append("normal"); path.append("stage1_normal")
            elif p_atk[i] >= ts:
                final.append(pred_cat[i]); path.append("stage2_strong")
            elif top1[i] < tc or marg[i] < tm:
                final.append("normal"); path.append("stage2_reverted")
            else:
                final.append(pred_cat[i]); path.append("stage2_confident")

        result = holdout_test_df.copy()
        result["stage1_p_attack"] = p_atk
        result["stage2_pred_category"] = pred_cat
        result["stage2_top1_conf"] = top1
        result["stage2_margin"] = marg
        result["final_prediction"] = final
        result["decision_path"] = path

        if "label" in holdout_test_df.columns:
            yt = to_binary(holdout_test_df["label"])
            yp = np.array([0 if v == "normal" else 1 for v in final])
            tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
            acc = accuracy_score(yt, yp)
            ar = tp / (tp + fn) if (tp + fn) else 0
            fpr = fp / (fp + tn) if (fp + tn) else 0

            print("\nSA BEST HOLDOUT TEST RESULTS")
            print(f"  Accuracy     : {acc*100:.2f}%")
            print(f"  Attack Recall: {ar*100:.2f}%")
            print(f"  FPR Normal   : {fpr*100:.4f}%")
            print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

        out_csv = os.path.join("data", "test", "KDDTrain_holdout_sa_best_randomforest.csv")
        result.to_csv(out_csv, index=False)
        print(f"\nSA best predictions saved -> {out_csv}")
else:
    print("Skipping SA test evaluation (RUN_TEST=False)")



SA BEST HOLDOUT TEST RESULTS
  Accuracy     : 99.80%
  Attack Recall: 99.69%
  FPR Normal   : 0.1039%
  TP=11690  FP=14  FN=36  TN=13455

SA best predictions saved -> data\test\KDDTrain_holdout_sa_best_randomforest.csv


## 📊 Threshold Tuning — Grid Search for Cascade Decision Boundaries



In [34]:
if RUN_TRAIN:
    t1_low_grid    = np.arange(CASCADE_TH["stage1_low"]["start"],
                               CASCADE_TH["stage1_low"]["stop"],
                               CASCADE_TH["stage1_low"]["step"])
    t1_strong_grid = np.arange(CASCADE_TH["stage1_strong"]["start"],
                               CASCADE_TH["stage1_strong"]["stop"],
                               CASCADE_TH["stage1_strong"]["step"])
    t2_conf_grid   = np.arange(CASCADE_TH["stage2_conf"]["start"],
                               CASCADE_TH["stage2_conf"]["stop"],
                               CASCADE_TH["stage2_conf"]["step"])
    t2_margin_grid = np.arange(CASCADE_TH["stage2_margin"]["start"],
                               CASCADE_TH["stage2_margin"]["stop"],
                               CASCADE_TH["stage2_margin"]["step"])

    min_recall = GENERAL["min_attack_recall_for_threshold_search"]

    # Stage 1 probabilities on the held-out val set (clean — not used for calibration)
    p_atk = cal1.predict_proba(X_v1)[:,1]

    # ── FIX: Only run Stage 2 on samples that could ever reach it ──
    # Any sample with p_atk < min(t1_low_grid) is always classified normal,
    # so stage 2 predictions are never used for those.
    min_tl = float(t1_low_grid.min())
    s2_mask = p_atk >= min_tl
    top1   = np.zeros(len(X_v1))
    margin = np.zeros(len(X_v1))

    if s2_mask.any():
        p_cat_s2 = cal2.predict_proba(X_v1[s2_mask])
        top1_s2  = np.max(p_cat_s2, axis=1)
        top2_s2  = np.partition(p_cat_s2, -2, axis=1)[:,-2]
        top1[s2_mask]   = top1_s2
        margin[s2_mask] = top1_s2 - top2_s2
    print(f"Stage 2 applied to {s2_mask.sum()}/{len(X_v1)} val samples")

    rows = []
    for tl in t1_low_grid:
        for ts in t1_strong_grid:
            if ts <= tl: continue
            for tc in t2_conf_grid:
                for tm in t2_margin_grid:
                    fb = np.zeros(len(X_v1), dtype=int)
                    fb[p_atk >= ts] = 1
                    weak = (p_atk >= tl) & (p_atk < ts)
                    fb[weak & (top1 >= tc) & (margin >= tm)] = 1

                    tn,fp,fn,tp = confusion_matrix(y_v1, fb, labels=[0,1]).ravel()
                    ar = tp/(tp+fn) if (tp+fn) else 0
                    fpr = fp/(fp+tn) if (fp+tn) else 0
                    bf1 = f1_score(y_v1, fb, zero_division=0)
                    rows.append(dict(stage1_low=round(float(tl),4), stage1_strong=round(float(ts),4),
                                     stage2_conf=round(float(tc),4), stage2_margin=round(float(tm),4),
                                     attack_recall=ar, fpr_normal=fpr, binary_f1=bf1,
                                     accuracy=accuracy_score(y_v1,fb)))

    # Select best
    valid = [r for r in rows if r["attack_recall"] >= min_recall]
    if valid:
        best = min(valid, key=lambda r: (r["fpr_normal"], -r["attack_recall"]))
    else:
        best = max(rows, key=lambda r: (r["attack_recall"], -r["fpr_normal"]))

    cascade_cfg = {
        "model": SELECTED_MODEL,
        "thresholds": {k: best[k] for k in ["stage1_low","stage1_strong","stage2_conf","stage2_margin"]},
        "validation_metrics": {k: best[k] for k in ["attack_recall","fpr_normal","binary_f1","accuracy"]}
    }

    with open(os.path.join(out_dir,"cascade_config.json"),"w") as f:
        json.dump(cascade_cfg, f, indent=2)

    print(f"\n{'='*55}")
    print(f"  BEST THRESHOLDS \u2014 {SELECTED_MODEL.upper()}")
    print(f"{'='*55}")
    for k,v in cascade_cfg["thresholds"].items():
        print(f"  {k:20s}: {v}")
    print()
    for k,v in cascade_cfg["validation_metrics"].items():
        print(f"  {k:20s}: {v:.6f}")
    print(f"\n\u2713 cascade_config.json saved \u2192 {out_dir}")
    print(f"  (searched {len(rows)} threshold combos)")
else:
    print("\u23ed Skipping threshold tuning (RUN_TRAIN=False)")



Stage 2 applied to 4700/10078 val samples

  BEST THRESHOLDS — XGBOOST
  stage1_low          : 0.7
  stage1_strong       : 0.75
  stage2_conf         : 0.3
  stage2_margin       : 0.05

  attack_recall       : 0.997015
  fpr_normal          : 0.000928
  binary_f1           : 0.997972
  accuracy            : 0.998115

✓ cascade_config.json saved → model\cascade\xgboost
  (searched 7020 threshold combos)


## 🧭 Baseline vs SA Comparison

This compares validation metrics for:
- Baseline (current RF params + grid-search thresholds, if available)
- SA best (from the saved SA config)

Outputs a compact comparison table and saves it to CSV.


In [35]:
if RUN_TRAIN:
    model_name = "randomforest"
    rf_out_dir = os.path.join("model", "cascade", model_name)
    space_map_path = os.path.join(rf_out_dir, "sa_search_space_randomforest.json")
    sa_best_path = os.path.join(rf_out_dir, "sa_best_config.json")
    grid_path = os.path.join(rf_out_dir, "cascade_config.json")

    def _build_rf_pipeline(params, X):
        pre = make_preprocessor(X)
        clf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            class_weight=params["class_weight"],
            random_state=GENERAL["random_state"],
            n_jobs=-1,
        )
        return Pipeline([("preprocessor", pre), ("classifier", clf)])

    def _calibrate(base, X_cal, y_cal, method):
        try:
            from sklearn.frozen import FrozenEstimator
            cal = CalibratedClassifierCV(FrozenEstimator(base), method=method, cv=None)
        except Exception:
            cal = CalibratedClassifierCV(base, method=method, cv="prefit")
        cal.fit(X_cal, y_cal)
        return cal

    def _evaluate(cfg):
        X_tr1 = train_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_tr1 = to_binary(train_df["label"])
        X_v1 = val_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_v1 = to_binary(val_df["label"])

        pipe1 = _build_rf_pipeline(cfg["rf_stage1"], X_tr1)
        pipe1.fit(X_tr1, y_tr1)

        y_tr_bin = to_binary(train_df["label"])
        atk_tr = train_df[y_tr_bin == 1].copy()
        atk_tr["attack_category"] = to_category(atk_tr["label"])
        X_tr2 = atk_tr.drop(columns=["label", "attack_category"], errors="ignore")
        le2 = LabelEncoder()
        y_tr2 = le2.fit_transform(atk_tr["attack_category"])

        pipe2 = _build_rf_pipeline(cfg["rf_stage2"], X_tr2)
        classes, counts = np.unique(y_tr2, return_counts=True)
        cw = {c: len(y_tr2) / (len(classes) * n) for c, n in zip(classes, counts)}
        sw = np.array([cw[v] for v in y_tr2])
        pipe2.fit(X_tr2, y_tr2, classifier__sample_weight=sw)

        method = GENERAL["calibration_method"]
        X_cal1 = cal_df.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal1 = to_binary(cal_df["label"])
        cal1 = _calibrate(pipe1, X_cal1, y_cal1, method)

        atk_cal = cal_df[to_binary(cal_df["label"]) == 1].copy()
        atk_cal["attack_category"] = to_category(atk_cal["label"])
        X_cal2 = atk_cal.drop(columns=["label", "attack_category"], errors="ignore")
        y_cal2 = le2.transform(atk_cal["attack_category"])
        cal2 = _calibrate(pipe2, X_cal2, y_cal2, method)

        p_atk = cal1.predict_proba(X_v1)[:, 1]
        tl = cfg["thresholds"]["stage1_low"]
        ts = cfg["thresholds"]["stage1_strong"]
        tc = cfg["thresholds"]["stage2_conf"]
        tm = cfg["thresholds"]["stage2_margin"]

        s2_mask = p_atk >= tl
        pred_cat = np.full(len(X_v1), "", dtype=object)
        top1 = np.zeros(len(X_v1))
        marg = np.zeros(len(X_v1))

        if s2_mask.any():
            p_cat = cal2.predict_proba(X_v1[s2_mask])
            top1[s2_mask] = np.max(p_cat, axis=1)
            top2_vals = np.partition(p_cat, -2, axis=1)[:, -2]
            marg[s2_mask] = top1[s2_mask] - top2_vals
            pred_cat[s2_mask] = le2.inverse_transform(np.argmax(p_cat, axis=1))

        final = []
        for i in range(len(X_v1)):
            if p_atk[i] < tl:
                final.append("normal")
            elif p_atk[i] >= ts:
                final.append(pred_cat[i])
            elif top1[i] < tc or marg[i] < tm:
                final.append("normal")
            else:
                final.append(pred_cat[i])

        yt = y_v1
        yp = np.array([0 if v == "normal" else 1 for v in final])
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        recall = tp / (tp + fn) if (tp + fn) else 0.0
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        accuracy = (tp + tn) / len(yt) if len(yt) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        fnr = fn / (fn + tp) if (fn + tp) else 0.0

        return {
            "recall": float(recall),
            "fpr": float(fpr),
            "precision": float(precision),
            "accuracy": float(accuracy),
            "fnr": float(fnr),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        }

    baseline_cfg = {
        "rf_stage1": dict(PARAMS[model_name]["stage1"]),
        "rf_stage2": dict(PARAMS[model_name]["stage2"]),
        "thresholds": {
            "stage1_low": float(CASCADE_TH["stage1_low"]["start"]),
            "stage1_strong": float(CASCADE_TH["stage1_strong"]["start"]),
            "stage2_conf": float(CASCADE_TH["stage2_conf"]["start"]),
            "stage2_margin": float(CASCADE_TH["stage2_margin"]["start"]),
        },
    }

    if os.path.exists(grid_path):
        with open(grid_path) as f:
            grid_cfg = json.load(f)
        baseline_cfg["thresholds"] = dict(grid_cfg.get("thresholds", baseline_cfg["thresholds"]))

    results = []
    results.append({"label": "baseline_grid", **_evaluate(baseline_cfg)})

    if os.path.exists(sa_best_path):
        with open(sa_best_path) as f:
            sa_cfg = json.load(f)
        results.append({"label": "sa_best", **_evaluate(sa_cfg["best_config"])})

    compare_df = pd.DataFrame(results)
    display(compare_df)

    compare_path = os.path.join(rf_out_dir, "sa_vs_baseline_comparison.csv")
    compare_df.to_csv(compare_path, index=False)
    print(f"Comparison saved -> {compare_path}")
else:
    print("Skipping comparison (RUN_TRAIN=False)")


,label,recall,fpr,precision,accuracy,fnr,tp,fp,fn,tn
0,baseline_grid,0.996588,0.000928,0.998931,0.997916,0.003412,4674,5,16,5383
1,sa_best,0.997441,0.001299,0.998506,0.998115,0.002559,4678,7,12,5381


Comparison saved -> model\cascade\randomforest\sa_vs_baseline_comparison.csv


## 🧪 Test Inference — Full Cascade on Test Dataset



In [36]:
if RUN_TEST:
    # Load models
    out_dir = os.path.join("model", "cascade", SELECTED_MODEL.lower())
    s1 = joblib.load(os.path.join(out_dir, "stage1_calibrated.pkl"))
    s2 = joblib.load(os.path.join(out_dir, "stage2_calibrated.pkl"))
    le = joblib.load(os.path.join(out_dir, "stage2_label_encoder.pkl"))
    with open(os.path.join(out_dir, "cascade_config.json")) as f:
        cfg = json.load(f)
    th = cfg["thresholds"]

    X_test = holdout_test_df.drop(columns=["label", "attack_category"], errors="ignore")
    print(f"Holdout test data: {len(holdout_test_df)} rows")

    # Stage 1: binary attack probability for all samples
    p_atk = s1.predict_proba(X_test)[:, 1]
    tl, ts = th["stage1_low"], th["stage1_strong"]
    tc, tm = th["stage2_conf"], th["stage2_margin"]

    # ── FIX: Only run Stage 2 on samples that need it (p_atk >= tl) ──
    s2_mask = p_atk >= tl
    pred_cat = np.full(len(X_test), "", dtype=object)
    top1 = np.zeros(len(X_test))
    marg = np.zeros(len(X_test))

    if s2_mask.any():
        X_s2 = X_test[s2_mask]
        p_cat = s2.predict_proba(X_s2)
        top1[s2_mask] = np.max(p_cat, axis=1)
        top2_vals = np.partition(p_cat, -2, axis=1)[:, -2]
        marg[s2_mask] = top1[s2_mask] - top2_vals
        pred_cat[s2_mask] = le.inverse_transform(np.argmax(p_cat, axis=1))
    print(f"  Stage 2 applied to {s2_mask.sum()}/{len(X_test)} samples")

    final, path = [], []
    for i in range(len(X_test)):
        if p_atk[i] < tl:
            final.append("normal"); path.append("stage1_normal")
        elif p_atk[i] >= ts:
            final.append(pred_cat[i]); path.append("stage2_strong")
        elif top1[i] < tc or marg[i] < tm:
            final.append("normal"); path.append("stage2_reverted")
        else:
            final.append(pred_cat[i]); path.append("stage2_confident")

    result = holdout_test_df.copy()
    result["stage1_p_attack"] = p_atk
    result["stage2_pred_category"] = pred_cat
    result["stage2_top1_conf"] = top1
    result["stage2_margin"] = marg
    result["final_prediction"] = final
    result["decision_path"] = path

    if "label" in holdout_test_df.columns:
        yt = to_binary(holdout_test_df["label"])
        yp = np.array([0 if v == "normal" else 1 for v in final])
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        acc = accuracy_score(yt, yp)
        ar = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0

        print(f"\n{'='*55}")
        print(f"  CASCADE HOLDOUT TEST RESULTS — {SELECTED_MODEL.upper()}")
        print(f"{'='*55}")
        print(f"  Accuracy     : {acc*100:.2f}%")
        print(f"  Attack Recall: {ar*100:.2f}%")
        print(f"  FPR Normal   : {fpr*100:.4f}%")
        print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

    print(f"\n  Decision path breakdown:")
    for p_name, cnt in pd.Series(path).value_counts().items():
        print(f"    {p_name:25s}: {cnt}")

    out_csv = os.path.join("data", "test", "KDDTrain_holdout_cascade.csv")
    result.to_csv(out_csv, index=False)
    print(f"\n\u2713 Predictions saved \u2192 {out_csv}")
else:
    print("\u23ed Skipping test inference (RUN_TEST=False)")


Holdout test data: 25195 rows
  Stage 2 applied to 11714/25195 samples

  CASCADE HOLDOUT TEST RESULTS — XGBOOST
  Accuracy     : 99.87%
  Attack Recall: 99.81%
  FPR Normal   : 0.0742%
  TP=11704  FP=10  FN=22  TN=13459

  Decision path breakdown:
    stage1_normal            : 13481
    stage2_strong            : 11714

✓ Predictions saved → data\test\KDDTrain_holdout_cascade.csv


## 📈 Quick Single-Sample Inspection



In [37]:
if RUN_TEST:
    sample = result.sample(5, random_state=42)[["label","final_prediction","decision_path",
              "stage1_p_attack","stage2_pred_category","stage2_top1_conf","stage2_margin"]]
    display(sample)
else:
    print("Run test inference first")



,label,final_prediction,decision_path,stage1_p_attack,stage2_pred_category,stage2_top1_conf,stage2_margin
13984,normal,normal,stage1_normal,0.001842,,0.000000,0.000000
17498,nmap,Probe,stage2_strong,0.999068,Probe,0.997037,0.995114
6455,normal,normal,stage1_normal,0.001847,,0.000000,0.000000
3482,normal,normal,stage1_normal,0.001845,,0.000000,0.000000
9252,teardrop,DoS,stage2_strong,0.999066,DoS,0.998494,0.997678
